# SymbolicTensors — Manifold, tangent bundle and frame bundle

This notebook explains the structures defined with @def_manifold macro and their hierarchy 

---
## 1. Loading package

In [1]:
using SymbolicTensors

---
## 2. Manifolds and def_manifold macro
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:tangentM, :cotangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension (concrete `Int` or symbolic `Symbol`)
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : names of all vector bundles over this manifold


In [3]:
?@def_manifold

```julia
@def_manifold <name> dim coord_indices frame_indices [kwargs...]
```

Define a new manifold and automatically create its tangent and cotangent bundles, coordinate frames, and moving frame bundles.

Both index lists are **required**. Each list should have at least 4 symbols (a warning is issued if fewer).

`dim` accepts a concrete positive integer or a bare symbol (e.g. `d`) for parametric / general-dimension calculations where the dimension is not fixed at definition time.

### Bindings in the caller's scope

  * `<name>`            → [`Manifold`](@ref) instance
  * `tangent<name>`     → [`VBundle`](@ref) (`isdual = false`)
  * `cotangent<name>`   → [`VBundle`](@ref) (`isdual = true`)
  * `frame<name>`       → [`FrameBundle`](@ref) (moving frame bundle)
  * `coframe<name>`     → [`FrameBundle`](@ref) (moving coframe bundle)
  * coordinate frame / coframe symbols (default `∂`, `dx`)
  * moving frame / coframe symbols (default `e`, `θ`)
  * each symbol in `coord_indices` → [`CoordinateIndex`](@ref) (contravariant)
  * each symbol in `frame_indices` → [`FrameIndex`](@ref) (contravariant)

### Index variance

Coordinate indices:

```julia
a1          # CoordinateIndex(:a1, :tangentM)   — contravariant
-a1         # CoordinateIndex(:a1, :cotangentM) — covariant
```

Frame indices:

```julia
A1          # FrameIndex(:A1, :tangentM)   — contravariant
-A1         # FrameIndex(:A1, :cotangentM) — covariant
```

### Keyword arguments

| Keyword          | Default | Description                                                 |
|:---------------- |:------- |:----------------------------------------------------------- |
| `coord_frame`    | `:∂`    | Basis name for the coordinate frame on the tangent bundle   |
| `coord_coframe`  | `:dx`   | Basis name for the coordinate frame on the cotangent bundle |
| `moving_frame`   | `:e`    | Basis name for the moving frame on the tangent bundle       |
| `moving_coframe` | `:θ`    | Basis name for the moving frame on the cotangent bundle     |

All four names must be distinct.

### Examples

```julia
# Minimal — concrete dimension, default frame names
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
# Binds: M, tangentM, cotangentM, frameM, coframeM,
#        ∂, dx (coordinate frames), e, θ (moving frames),
#        a1, a2, a3, a4 (CoordinateIndex), A1, A2, A3, A4 (FrameIndex)

# Parametric dimension
@def_manifold M d [a1, a2, a3, a4] [A1, A2, A3, A4]

# Custom frame names — all four must be distinct
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]   \
    coord_frame=:e_coord   coord_coframe=:θ_coord       \
    moving_frame=:e_mov    moving_coframe=:θ_mov
```


In [4]:
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]

Defined manifold M of dimension 4
Defined coordinate frame ∂ on tangentM and coordinate coframe dx on cotangentM
Defined frame bundle frameM (moving frame e) and coframe bundle coframeM (moving coframe θ) over M


In [5]:
M

Dimension,4
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


In [6]:
# dot access to a Manifold properties 
(M.name, M.dim, M.tangent_bundle, M.cotangent_bundle, M.vbundles)

(:M, 4, :tangentM, :cotangentM, [:tangentM, :cotangentM])

In [7]:
# Manifold registry
(_MANIFOLDS,M==_MANIFOLDS[:M])

(Dict{Symbol, Manifold}(:M => Manifold(M, dim=4, TBundle=tangentM, CBundle=cotangentM)), true)

In [8]:
@def_manifold N 4 [b1, b2, b3, b4] [B1, B2, B3, B4] coord_frame = d coord_coframe = dx 

Defined manifold N of dimension 4
Defined coordinate frame d on tangentN and coordinate coframe dx on cotangentN
Defined frame bundle frameN (moving frame e) and coframe bundle coframeN (moving coframe θ) over N


In [48]:
dx[b1]

dx[b1]

In [10]:
d[-b1]

d[-b1]

### b. (co)Tangent bundle and (co)Frame bundle 

In [11]:
@doc VBundle

```julia
VBundle
```

Struct representing a registered vector bundle. Instances are created by [`@def_manifold`](@ref) for the tangent and cotangent bundles, and by [`@def_vbundle`](@ref) for custom bundles. Bound to variables in the caller's scope (e.g. `tangentM`, `cotangentM`).

Provides dot access to all metadata:

```julia
tangentM.name             # :tangentM
tangentM.manifold         # :M
tangentM.dim              # 4
tangentM.isdual           # false
tangentM.dual             # :cotangentM
tangentM.coordinate_indices  # [CoordinateIndex(:a1, :tangentM), ...]
tangentM.frame_indices       # [FrameIndex(:A1, :tangentM), ...]
tangentM.bases            # [Basis(:∂, :tangentM, :coordinate),
                          #  Basis(:e, :tangentM, :frame)]
```

### Fields

  * `name`               : bundle name, e.g. `:tangentM`
  * `manifold`           : base manifold name, e.g. `:M`
  * `dim`                : fibre dimension
  * `isdual`             : `false` = primal (contravariant), `true` = dual                        (covariant). Authoritative for variance via                        [`is_up`](@ref) / [`is_down`](@ref).
  * `dual`               : name of the paired dual bundle
  * `coordinate_indices` : [`CoordinateIndex`](@ref) objects for the coordinate                        chart; nonempty for tangent/cotangent bundles
  * `frame_indices`      : [`FrameIndex`](@ref) objects for the fibre frame;                        populated by [`@def_manifold`](@ref) and                        [`@def_vbundle`](@ref)


In [12]:
tangentM

VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4, bases=[Basis(∂, tangentM, coord), Basis(e, tangentM, frame)])

In [13]:
(tangentM.bases,tangentM.coordinate_indices,tangentM.basis_indices)

LoadError: FieldError: type VBundle has no field `basis_indices`, available fields: `name`, `manifold`, `dim`, `isdual`, `dual`, `coordinate_indices`, `frame_indices`
Available properties: `bases`

In [14]:
(cotangentM,tangentM)

(VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=4, bases=[Basis(dx, cotangentM, coord), Basis(θ, cotangentM, frame)]), VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4, bases=[Basis(∂, tangentM, coord), Basis(e, tangentM, frame)]))

In [15]:
(cotangentM.bases,cotangentM.coordinate_indices,cotangentM.basis_indices)

LoadError: FieldError: type VBundle has no field `basis_indices`, available fields: `name`, `manifold`, `dim`, `isdual`, `dual`, `coordinate_indices`, `frame_indices`
Available properties: `bases`

In [16]:
@doc Basis

```julia
Basis
```

A named frame for a vector bundle, of a given category (`:coordinate` or `:frame`). Instances are created by [`@def_manifold`](@ref) (coordinate and frame) or standalone [`@def_frame_bundle`](@ref) (frame only for custom bundles).

### Fields

  * `name`     : display name, e.g. `:dx`, `:∂`, `:e`, `:θ`
  * `vbundle`  : the bundle this frame is for, e.g. `:cotangentM`
  * `category` : `:coordinate` (coordinate-induced) or `:frame` (arbitrary chosen local frame)

Indexing a `Basis` with an [`AbstractIndex`](@ref) from the **dual** bundle produces a [`BasisElement`](@ref):

```julia
dx[a1]    # a1 ∈ tangentM  → BasisElement of cotangentM coordinate frame
e[-a1]    # -a1 ∈ cotangentM → BasisElement of tangentM arbitrary frame
```


In [17]:
cotangentM.bases

2-element Vector{Basis}:
 Basis(dx, cotangentM, coord)
 Basis(θ, cotangentM, frame)

In [18]:
(dx,θ)

(Basis(dx, cotangentN, coord), Basis(θ, cotangentN, frame))

In [19]:
tangentM.bases

2-element Vector{Basis}:
 Basis(∂, tangentM, coord)
 Basis(e, tangentM, frame)

In [20]:
(∂,e)

(Basis(∂, tangentM, coord), Basis(e, tangentN, frame))

In [21]:
@doc BasisElement

```julia
BasisElement
```

A single element of a basis (coordinate or frame), constructed by `getindex` on a [`Basis`](@ref).

### Fields

  * `basis` : the [`Basis`](@ref) this element belongs to
  * `index` : the [`AbstractIndex`](@ref) labeling this element;           its vbundle is the **dual** of `basis.vbundle`

    dx[a1]   → BasisElement(Basis(:dx, :cotangentM, :coordinate), CoordinateIndex(:a1, :tangentM))   θ[A1]    → BasisElement(Basis(:θ,  :cotangentM, :frame),      BasisIndex(:A1, :tangentM))   ∂[-a1]   → BasisElement(Basis(:∂,  :tangentM,   :coordinate), CoordinateIndex(:a1, :cotangentM))   e[-A1]   → BasisElement(Basis(:e,  :tangentM,   :frame),      BasisIndex(:A1, :cotangentM))


In [22]:
(dx[a1],∂[-a1])

LoadError: BasisElement: index vbundle :tangentM is not the dual of basis vbundle :cotangentN. Expected index from :tangentN. Use dx[...] with an index from the dual bundle.

In [23]:
(θ[A1],e[-A1])

LoadError: BasisElement: index vbundle :tangentM is not the dual of basis vbundle :cotangentN. Expected index from :tangentN. Use θ[...] with an index from the dual bundle.

#### Error handling 

In [24]:
dx[-a1]

LoadError: BasisElement: index vbundle :cotangentM is not the dual of basis vbundle :cotangentN. Expected index from :tangentN. Use dx[...] with an index from the dual bundle.

In [25]:
dx[A1]

LoadError: BasisElement: index vbundle :tangentM is not the dual of basis vbundle :cotangentN. Expected index from :tangentN. Use dx[...] with an index from the dual bundle.

#### Change of basis coordinate <-> moving

### c. Frame bundle 

In [26]:
frameM

VBundle,tangentM
Dual,coframeM
Frame basis,e


In [27]:
coframeM

VBundle,cotangentM
Dual,frameM
Frame basis,θ


---
## 3. Tensors
---

### a. Type, definition and tensor components

In [28]:
@doc Tensor

```julia
Tensor
```

A registered abstract tensor. Instances are created by [`@def_tensor`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
T.manifold       # :M  (Symbol key into _MANIFOLDS)
T.slots          # [:cotangentM, :cotangentM]  — vbundle per slot
T.symmetries     # [SlotSymmetry(n=2, order=2)]
T.is_traceless   # false
T.known_traces   # Any[]  (populated later)
T.print_as       # :T
T.metric         # :g or nothing
T.rank           # 2      (derived — length of slots)
```

### Fields

  * `manifold`      : name of the base manifold, key into `_MANIFOLDS`
  * `slots`         : vbundle symbol per slot, encoding variance.                   `:cotangentM` → covariant, `:tangentM` → contravariant.
  * `symmetries`    : `Vector{`[`SlotSymmetry`](@ref)`}` — list of permutation                   groups acting on slot positions `1:rank`.
  * `is_traceless`  : if `true`, any self-contraction of this tensor gives zero                   (e.g. the Weyl tensor).
  * `known_traces`  : user-declared trace values, e.g. `g[a,-a] = dim`.                   Format TBD — stored as `Any[]` until contraction is                   implemented.
  * `print_as`      : display name. Controls how the tensor appears in `show`                   and (later) LaTeX output.
  * `metric`        : name of the metric tensor associated with this tensor,                   a key into `_METRICS`, or `nothing` if no metric                   has been assigned. Required for raising/lowering indices.


In [29]:
@doc @def_tensor

```julia
@def_tensor name [vbundle1, vbundle2, ...] [symmetries=...] [traceless=...] [print_as=:...] [metric=...]
```

Define a new abstract tensor and bind it to `name` in the caller's scope.

### Slot syntax

Specify slots directly as VBundle symbols. The manifold is automatically deduced from the first VBundle's `manifold` field.

  * `:tangentM` → contravariant (upper) slot
  * `:cotangentM` → covariant (lower) slot
  * Any user-defined VBundle from `@def_vbundle`

All VBundle symbols must belong to the same manifold.

### Keyword arguments (all optional)

  * `symmetries`  : a [`SlotSymmetry`](@ref) or `Vector{SlotSymmetry}` describing                 the slot permutation symmetry group(s). Defaults to                 `[no_symmetry(rank)]`.
  * `traceless`   : `true` if any self-contraction of this tensor is zero                 (e.g. Weyl tensor). Defaults to `false`.
  * `print_as`    : display name. Defaults to `name`. Example: `print_as=:R`.
  * `metric`      : name of the metric tensor to associate with this tensor.                 Omitting this keyword triggers automatic resolution:                 - one metric on manifold → assigned silently                 - multiple metrics → `@warn`, first defined is assigned                 - no metric → `@warn`, `nothing` assigned (no raising/lowering)

Binds `name` to a [`Tensor`](@ref) instance in the caller's scope and registers it in [`_TENSORS`](@ref).

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]
@def_metric g M

@def_tensor T [cotangentM, cotangentM]                 # (0,2) tensor, metric auto-resolved
@def_tensor F [cotangentM, cotangentM] symmetries=[antisymmetric(2)]
@def_tensor R [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()]
@def_tensor W [cotangentM, cotangentM, cotangentM, tangentM] symmetries=[riemann_symmetry()] traceless=true print_as=:Weyl
@def_tensor mixed_T [tangentM, cotangentM]            # (1,1) mixed tensor
```


In [30]:
cotangentM

VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=4, bases=[Basis(dx, cotangentM, coord), Basis(θ, cotangentM, frame)])

In [31]:
_VBUNDLES

Dict{Symbol, VBundle} with 4 entries:
  :cotangentM => VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=…
  :tangentM   => VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4,…
  :cotangentN => VBundle(cotangentN, cotangent, dual=tangentN, manifold=N, dim=…
  :tangentN   => VBundle(tangentN, tangent, dual=cotangentN, manifold=N, dim=4,…

In [32]:
@def_tensor T [cotangentM, cotangentM]

┌ Warning: No metric is defined on manifold M. Tensor T has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:325


Defined tensor T on manifold M with 2 slot(s), metric=nothing


In [33]:
T

Manifold,M
Rank,2
Slots,↓cotangentM ↓cotangentM
Symmetries,NoSymmetry(n=2)
Traceless,false
Metric,none


In [34]:
(T.is_traceless,T.known_traces,T.manifold,T.metric,T.print_as,T.rank,T.slots,T.symmetries)

(false, Any[], :M, nothing, :T, 2, [:cotangentM, :cotangentM], SlotSymmetry[NoSymmetry(n=2)])

In [35]:
T.symmetries

1-element Vector{SlotSymmetry}:
 NoSymmetry(n=2)

In [36]:
T.slots

2-element Vector{Symbol}:
 :cotangentM
 :cotangentM

In [37]:
T[-a1,-a2]

T[-a1, -a2]

In [38]:
(T[-a1,-a2],T[a1,-a2],T[-a1,a2],T[a1,a2],T[-A1,-A2],T[A1,-a1],T[A1,a1])

(T[-a1, -a2], T[a1, -a2], T[-a1, a2], T[a1, a2], T[-A1, -A2], T[A1, -a1], T[A1, a1])

##### Error handling 

In [39]:
T[-a1,-a1]

T[-a1, -a1]

In [40]:
T[a1,a1]

T[a1, a1]

### b. Expansion of a tensor within a basis 

In [41]:
@doc basis_expansion

```julia
basis_expansion(T::Tensor, style::ExpansionStyle=Coordinate) -> BasisExpansion
basis_expansion(T::Tensor) -> BasisExpansion
```

Expand tensor `T` slot-by-slot using canonical indices and the given [`ExpansionStyle`](@ref).

  * **`Coordinate`** (default): per slot use `:coordinate` basis if registered, else `:frame`.
  * **`Frame`**: per slot use `:frame` basis only.

# Examples

```julia
basis_expansion(T)             # Coordinate style
basis_expansion(T, Coordinate)
basis_expansion(T, Frame)
```


In [42]:
basis_expansion(T)

T[-a1, -a2] dx[a1] ⊗ dx[a2]

In [43]:
(basis_expansion(T, Coordinate),basis_expansion(T, Frame))

LoadError: UndefVarError: `_BASIS_INDICES` not defined in `SymbolicTensors`
Suggestion: check for spelling errors or missing imports.

In [44]:
@def_tensor B [tangentM, cotangentM]

Defined tensor B on manifold M with 2 slot(s), metric=nothing


┌ Warning: No metric is defined on manifold M. Tensor B has no metric assigned; indices cannot be raised or lowered.
└ @ Main ~/JuliaTensor/SymbolicTensors/src/tensors.jl:325


In [45]:
(basis_expansion(B,Coordinate),basis_expansion(B, Frame))

LoadError: UndefVarError: `_BASIS_INDICES` not defined in `SymbolicTensors`
Suggestion: check for spelling errors or missing imports.